<a href="https://colab.research.google.com/github/Fenriro37/GRPO-Summarization/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [3]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.10.1", "triton==3.2.0") if is_t4 else ("vllm", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

In [1]:
from google.colab import drive
!pip install "wandb" "datasets" "evaluate" "rouge_score" "nltk" "tqdm"

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/dataset_with_len_filtered.json'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# @title Step 2: Imports and W&B Login
import json
import re
import torch
import wandb
import nltk
import numpy as np
from tqdm.notebook import tqdm  # Use notebook-friendly tqdm
from datasets import Dataset
from unsloth import FastLanguageModel
from vllm import SamplingParams
import evaluate

nltk.download('punkt_tab')


# --- Login to Weights & Biases ---
from huggingface_hub import notebook_login
import wandb
wandb.login(key='4c65a1c79b0c2cb47aaf9b96f87b38d2abd661b1')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 09-13 20:53:32 [__init__.py:241] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mattiamaranzana (mattiamaranzana-universit-di-bologna) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# @title Step 3: Configuration
# --- Model and Adapter Configuration ---
BASE_MODEL_NAME="HuggingFaceTB/SmolLM2-360M-Instruct"

HF_REPO_ID = "M4TT1A/my-llama3-grpo-lora"  # <--- CHANGE THIS to your HF repo ID
LORA_RANK = 32
MAX_SEQ_LENGTH = 4096

# If using Google Colab, you might need to mount your Drive first:
# from google.colab import drive
# drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/dataset_with_len_filtered.json'
BATCH_SIZE = 8

# --- W&B Configuration ---
WANDB_PROJECT = "grpo-summarization"
WANDB_RUN_NAME = "evaluation-run"

In [4]:
# @title Step 4: Metric and Utility Functions

def count_words(text):
    if not text: return 0
    return len(nltk.word_tokenize(text))

def count_sentences(text):
    if not text: return 0
    return len(nltk.sent_tokenize(text))

def extract_xml_answer(text: str) -> str:
    match = re.search(r"<summary>(.*?)</summary>", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

def reward_word_count_normalized(completions, target_word_count, tolerance=5, **kwargs):
    scores = []
    FORMAT_FAILURE_PENALTY = -1.0
    for completion, target_words in zip(completions, target_word_count):
        if target_words is None:
            scores.append(0.0)
            continue
        response_text = completion[0]["content"]
        summary_text = extract_xml_answer(response_text)
        if summary_text is None:
            scores.append(FORMAT_FAILURE_PENALTY)
            continue
        num_words = count_words(summary_text)
        distance_from_window = max(0, abs(num_words - target_words) - tolerance)
        score = -distance_from_window / target_words if target_words > 0 else FORMAT_FAILURE_PENALTY
        scores.append(score)
    return scores

def reward_sentence_count_normalized(completions, target_sentence_count, **kwargs):
    scores = []
    FORMAT_FAILURE_PENALTY = -1.0
    for completion, target_sentences in zip(completions, target_sentence_count):
        if target_sentences is None:
            scores.append(0.0)
            continue
        response_text = completion[0]["content"]
        summary_text = extract_xml_answer(response_text)
        if summary_text is None:
            scores.append(FORMAT_FAILURE_PENALTY)
            continue
        num_sentences = count_sentences(summary_text)
        distance = abs(num_sentences - target_sentences)
        score = -distance / target_sentences if target_sentences > 0 else FORMAT_FAILURE_PENALTY
        scores.append(score)
    return scores

def reward_for_structure_normalized(completions, **kwargs) -> list[float]:
    scores = []
    full_pattern = re.compile(r"^\s*<reasoning>.*?</reasoning>\s*<summary>.*?</summary>\s*$", re.DOTALL)
    for completion in completions:
        text = completion[0]["content"]
        if full_pattern.search(text):
            scores.append(0.0)
            continue
        penalty = 0.0
        if "<reasoning>" not in text: penalty -= 0.25
        if "</reasoning>" not in text: penalty -= 0.25
        if "<summary>" not in text: penalty -= 0.25
        if "</summary>" not in text: penalty -= 0.25
        temp_text = text.strip()
        if not temp_text.startswith("<reasoning>") or not temp_text.endswith("</summary>"):
             penalty -= 0.2
        scores.append(max(-1.0, penalty))
    return scores

In [5]:
def evaluate_model(model, tokenizer, test_dataset, lora_request, batch_size, temperature=0.1, top_p=1.0):
    """
    Runs evaluation on a given dataset, calculates all metrics, and returns
    the aggregated results and a W&B table for qualitative analysis.
    """
    sampling_params = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=1024)
    rouge = evaluate.load('rouge')

    results_data = []
    all_generated_summaries, all_gold_summaries = [], []
    all_word_scores, all_sentence_scores, all_structure_scores = [], [], []

    for i in tqdm(range(0, len(test_dataset), batch_size), desc="Evaluating Batches"):
        batch = test_dataset[i:i+batch_size]

        # Access the entire column (list of prompts) at once
        batch_prompts_data = batch['prompt']

        # Apply the chat template to each item in the list of prompts
        prompts = [
            tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)
            for chat_history in batch_prompts_data
        ]

        gold_summaries = batch['answer']
        target_words = batch['target_word_count']
        target_sentences = batch['target_sentence_count']

        outputs = model.fast_generate(prompts, sampling_params=sampling_params, lora_request=lora_request)
        raw_completions = [output.outputs[0].text for output in outputs]

        generated_summaries = [extract_xml_answer(comp) for comp in raw_completions]
        formatted_completions = [[{"content": comp}] for comp in raw_completions]

        word_scores = reward_word_count_normalized(formatted_completions, target_words)
        sentence_scores = reward_sentence_count_normalized(formatted_completions, target_sentences)
        structure_scores = reward_for_structure_normalized(formatted_completions)

        valid_pairs = [(gen, gold) for gen, gold in zip(generated_summaries, gold_summaries) if gen is not None]
        if valid_pairs:
            valid_generated, valid_gold = zip(*valid_pairs)
            all_generated_summaries.extend(valid_generated)
            all_gold_summaries.extend(valid_gold)

        all_word_scores.extend(word_scores)
        all_sentence_scores.extend(sentence_scores)
        all_structure_scores.extend(structure_scores)

        # Get the number of items in the batch from the length of one of its lists
        num_items_in_batch = len(batch['prompt'])
        for j in range(num_items_in_batch):
            results_data.append([
                prompts[j], gold_summaries[j], raw_completions[j],
                generated_summaries[j] or "FORMAT_ERROR",
                word_scores[j], sentence_scores[j], structure_scores[j]
            ])

    if not all_generated_summaries:
        print("Warning: No valid summaries were generated. ROUGE scores will be zero.")
        rouge_results = {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    else:
        rouge_results = rouge.compute(predictions=all_generated_summaries, references=all_gold_summaries)

    metrics = {
        "rouge1": rouge_results['rouge1'], "rouge2": rouge_results['rouge2'], "rougeL": rouge_results['rougeL'],
        "avg_word_count_score": np.mean(all_word_scores),
        "avg_sentence_count_score": np.mean(all_sentence_scores),
        "avg_structure_score": np.mean(all_structure_scores),
        "format_failure_rate": sum(1 for s in generated_summaries if s is None) / len(test_dataset)
    }

    results_table = wandb.Table(
        columns=["Prompt", "Gold Summary", "Raw Completion", "Generated Summary", "Word Score", "Sentence Score", "Structure Score"],
        data=results_data
    )

    return metrics, results_table

In [4]:
# --- W&B Setup ---
BASE_MODEL_NAME="HuggingFaceTB/SmolLM2-360M-Instruct"
file_path = '/content/drive/MyDrive/dataset_with_len_filtered.json'
HF_REPO_ID = "M4TT1A/my-llama3-grpo-lora"
LORA_RANK = 32
MAX_SEQ_LENGTH = 4096                               # Maximum sequence length for the model.
max_completion_length = 800
BATCH_SIZE = 4
config_dict = {
    "base_model_name": BASE_MODEL_NAME,
    "hf_repo_id": HF_REPO_ID,
    "lora_rank": LORA_RANK,
    "max_seq_length": MAX_SEQ_LENGTH,
    "dataset_path": file_path,
    "batch_size": BATCH_SIZE,
}
#wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME, config=config_dict)

# --- Load Dataset ---
print(f"Loading dataset from {file_path}...")
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

from datasets import DatasetDict
dataset_dict = DatasetDict({
    'train': Dataset.from_list(data['train']),
    'validation': Dataset.from_list(data['validation']),
    'test': Dataset.from_list(data['test'])
})

print(f"Loaded {len(dataset_dict['test'])} examples in the test set.")

Loading dataset from /content/drive/MyDrive/dataset_with_len_filtered.json...
Loaded 571 examples in the test set.


In [26]:
data.keys()

dict_keys(['train', 'validation', 'test'])

In [5]:
# @ Custom rewards
import numpy as np
import re
from nltk.tokenize import sent_tokenize, word_tokenize

def calculate_length_reward(generated_text, target_word_count, target_sentence_count, std_dev_factor=0.1):
    """
    Calculates a reward based on how close the summary length is to the target.
    A higher reward is better (max 1.0).
    std_dev_factor: Controls how sharply the reward drops off. Smaller values are stricter.
    """
    # Isolate the summary text
    summary_match = re.search(r'<summary>(.*?)</summary>', generated_text, re.DOTALL)
    if not summary_match:
        return 0.0 # Heavy penalty if the structure is wrong

    summary_text = summary_match.group(1).strip()
    if not summary_text:
        return 0.0 # Penalize empty summaries

    # Determine which constraint to use
    if target_word_count is not None:
        generated_count = len(word_tokenize(summary_text))
        target_count = target_word_count
        # Set a reasonable standard deviation based on the target length
        std_dev = max(1, target_count * std_dev_factor)

    elif target_sentence_count is not None:
        generated_count = len(sent_tokenize(summary_text))
        target_count = target_sentence_count
        # For sentences, the deviation is more sensitive
        std_dev = max(1, target_count * std_dev_factor * 0.5)
    else:
        return 1.0 # No constraint, no penalty

    # Calculate the Gaussian-based reward
    error = generated_count - target_count
    reward = np.exp(-0.5 * (error / std_dev)**2)

    return float(reward)

def calculate_structure_reward(generated_text):
    """
    Calculates a reward based on the presence and order of XML tags.
    A higher reward is better (max 1.0).
    """
    # Simple checks for presence and basic order
    has_reasoning = '<reasoning>' in generated_text and '</reasoning>' in generated_text
    has_summary = '<summary>' in generated_text and '</summary>' in generated_text

    if not has_reasoning and not has_summary:
        return 0.0

    if has_reasoning and has_summary:
        # Check order
        try:
            reasoning_start = generated_text.index('<reasoning>')
            summary_start = generated_text.index('<summary>')
            if reasoning_start < summary_start:
                return 1.0 # Perfect structure
            else:
                return 0.2 # Tags present but in wrong order
        except ValueError:
            return 0.1 # Should not happen if both tags are present

    # Only one of the two tags is present
    return 0.5


from rouge_score import rouge_scorer
import re

# Initialize the scorer once to be efficient
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def calculate_quality_reward(generated_text, reference_answer):
    """
    Calculates the ROUGE-L F1 score between the generated and reference summary.
    A higher reward is better (max 1.0).
    """
    summary_match = re.search(r'<summary>(.*?)</summary>', generated_text, re.DOTALL)
    if not summary_match:
        return 0.0 # No summary, no quality

    generated_summary = summary_match.group(1).strip()
    if not generated_summary:
        return 0.0

    # reference_answer is the 'answer' column from your dataset
    scores = scorer.score(reference_answer, generated_summary)

    return scores['rougeL'].fmeasure
def get_composite_reward(generation, example):
    """
    Calculates the total weighted reward for a single generation.
    'generation' is the full text output from the model.
    'example' is a dictionary row from your dataset.
    """
    # Weights - TUNE THESE!
    w_length = 0.5
    w_structure = 0.3
    w_quality = 0.2

    # Calculate individual reward components
    r_length = calculate_length_reward(
        generated_text=generation,
        target_word_count=example.get('target_word_count'),
        target_sentence_count=example.get('target_sentence_count')
    )

    r_structure = calculate_structure_reward(
        generated_text=generation
    )

    r_quality = calculate_quality_reward(
        generated_text=generation,
        reference_answer=example['answer']
    )

    # Calculate final weighted score
    total_reward = (w_length * r_length) + \
                   (w_structure * r_structure) + \
                   (w_quality * r_quality)

    return total_reward

In [6]:
from typing import List

# ==============================================================================
# Your original single-instance reward functions (no changes needed here)
# calculate_length_reward, calculate_structure_reward, calculate_quality_reward
# ... (I'll omit them for brevity, but assume they are defined as in your question)
# ==============================================================================

# NEW BATCH-COMPATIBLE WRAPPER FUNCTIONS

def length_reward_batch(completions: List[str], **kwargs) -> List[float]:
    """
    Calculates length rewards for a batch of completions.
    The trainer passes dataset columns as lists in kwargs.
    """
    # Extract the lists of targets from the kwargs passed by the trainer
    target_word_counts = kwargs["target_word_count"]
    target_sentence_counts = kwargs["target_sentence_count"]

    rewards = []
    for i, completion in enumerate(completions):
        # For each completion, get its corresponding target counts
        reward = calculate_length_reward(
            generated_text=completion,
            target_word_count=target_word_counts[i],
            target_sentence_count=target_sentence_counts[i]
        )
        rewards.append(reward)
    return rewards

def structure_reward_batch(completions: List[str], **kwargs) -> List[float]:
    """
    Calculates structure rewards for a batch of completions.
    This one is simpler as it doesn't need extra data.
    """
    return [calculate_structure_reward(c) for c in completions]

def quality_reward_batch(completions: List[str], **kwargs) -> List[float]:
    """
    Calculates ROUGE-L quality rewards for a batch of completions.
    """
    # Extract the list of reference answers from kwargs
    reference_answers = kwargs["answer"]

    rewards = []
    # Use zip to iterate through completions and their corresponding answers
    for completion, reference in zip(completions, reference_answers):
        reward = calculate_quality_reward(
            generated_text=completion,
            reference_answer=reference
        )
        rewards.append(reward)
    return rewards

In [7]:
max_seq_length = 4096                               # Maximum sequence length for the model.
max_completion_length = 800                         # Maximum completion length for the model.

# --- LoRA arguments ---
lora_rank = 16                                      # The rank for LoRA.

# --- Dataset arguments ---

# --- Training arguments ---
learning_rate = 5e-6                                # The learning rate for the optimizer.
max_steps = 5                                       # Total number of training steps.
batch_size = 1                                      # Per-device training batch size.
gradient_accumulation_steps = 4                     # Number of steps for gradient accumulation.
logging_steps = 10                                  # Log metrics every N steps.
save_steps = 50                                     # Save a checkpoint every N steps.
output_dir = "outputs"                              # Directory to save model checkpoints.
lora_dir = "lora_adapters"                          # Directory to save LoRA adapters.

# --- W&B and Hugging Face Hub arguments ---
wandb_project = "grpo-summarization"                # The Weights & Biases project name.
hf_repo_name = "M4TT1A/my-llama3-grpo-lora"          # The name of the repository on the Hugging Face Hub.
hf_repo_commit = None                               # A specific commit hash for the repo, if needed.

In [8]:
# --- Load Model and Tokenizer ---
print(f"Loading base model '{BASE_MODEL_NAME}'...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True, fast_inference=True, max_lora_rank=LORA_RANK,
    gpu_memory_utilization = 0.95
)


model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = LORA_RANK,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

Loading base model 'HuggingFaceTB/SmolLM2-360M-Instruct'...
Unsloth: Patching vLLM v1 graph capture
Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.9.4: Fast Llama patching. Transformers: 4.55.4. vLLM: 0.10.1.1.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading HuggingFaceTB/SmolLM2-360M-Instruct with actual GPU utilization = 94.46%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 79.32 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 4096. Num Sequences = 368.
Unsloth: vLLM's KV Cache can use up to 74.22 GB. Also swap space = 6 GB.
Unsloth: Not an error, but `device` is not supporte

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 09-13 20:54:06 [punica_selector.py:19] Using PunicaWrapperGPU.
INFO 09-13 20:54:07 [gpu_model_runner.py:2007] Model loading took 0.2807 GiB and 1.293302 seconds
INFO 09-13 20:54:24 [backends.py:548] Using cache directory: /root/.cache/vllm/torch_compile_cache/3c612ace14/rank_0_0/backbone for vLLM's torch.compile
INFO 09-13 20:54:24 [backends.py:559] Dynamo bytecode transform time: 16.15 s
INFO 09-13 20:54:35 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 8.937 s
INFO 09-13 20:54:38 [monitor.py:34] torch.compile takes 16.15 s in total
INFO 09-13 20:54:40 [gpu_worker.py:276] Available KV cache memory: 73.96 GiB
INFO 09-13 20:54:41 [kv_cache_utils.py:849] GPU KV cache size: 1,938,880 tokens
INFO 09-13 20:54:41 [kv_cache_utils.py:853] Maximum concurrency for 4,096 tokens per request: 473.36x
INFO 09-13 20:54:41 [vllm_utils.py:667] Unsloth: Running patched vLLM v1 `capture_model`.


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:28<00:00,  2.39it/s]

INFO 09-13 20:55:09 [gpu_model_runner.py:2708] Graph capturing finished in 28 secs, took 1.28 GiB
INFO 09-13 20:55:09 [vllm_utils.py:674] Unsloth: Patched vLLM v1 graph capture finished in 28 secs.


INFO 09-13 20:55:11 [core.py:214] init engine (profile, create kv cache, warmup model) took 63.93 seconds
INFO 09-13 20:55:12 [llm.py:298] Supported_tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['k_norm', 'pre_feedforward_layernorm', 'post_feedforward_layernorm', 'q_norm']
Unsloth: Just some info: will skip parsing ['k_norm', 'pre_feedforward_layernorm', 'post_feedforward_layernorm', 'q_norm']
HuggingFaceTB/SmolLM2-360M-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.


Unsloth 2025.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
# ==============================================================================
# Step 1: Import necessary libraries
# ==============================================================================
import re
import numpy as np
from typing import List

# Ensure NLTK data is downloaded (you only need to do this once)
import nltk
nltk.download('punkt_tab')

from nltk.tokenize import sent_tokenize, word_tokenize
from rouge_score import rouge_scorer

from trl import GRPOConfig, GRPOTrainer

# ==============================================================================
# Step 2: Define Reward Logic for a SINGLE Generation
# These are your modular, single-instance calculation functions.
# ==============================================================================

# Initialize the ROUGE scorer once to be efficient
rouge_scorer_instance = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def calculate_length_reward(generated_text: str, target_word_count: int, target_sentence_count: int, std_dev_factor=0.1) -> float:
    """
    Calculates a reward based on how close the summary length is to the target.
    A higher reward is better (max 1.0).
    std_dev_factor: Controls how sharply the reward drops off. Smaller values are stricter.
    """
    summary_match = re.search(r'<summary>(.*?)</summary>', generated_text, re.DOTALL)
    if not summary_match:
        return 0.0  # Heavy penalty if the structure is wrong

    summary_text = summary_match.group(1).strip()
    if not summary_text:
        return 0.0  # Penalize empty summaries

    # Determine which constraint to use (prioritizing word count if both are present)
    if target_word_count is not None and target_word_count > 0:
        generated_count = len(word_tokenize(summary_text))
        target_count = target_word_count
        std_dev = max(1, target_count * std_dev_factor)
    elif target_sentence_count is not None and target_sentence_count > 0:
        generated_count = len(sent_tokenize(summary_text))
        target_count = target_sentence_count
        std_dev = max(1, target_count * (std_dev_factor * 0.5)) # Sentences are more sensitive
    else:
        return 1.0  # No constraint, so no penalty

    # Calculate the Gaussian-based reward (peaks at 1.0 when error is 0)
    error = generated_count - target_count
    reward = np.exp(-0.5 * (error / std_dev)**2)
    return float(reward)

def calculate_structure_reward(generated_text: str) -> float:
    """
    Calculates a reward based on the presence and order of XML tags.
    A higher reward is better (max 1.0).
    """
    # Use a robust regex to check for the full, correct structure
    full_pattern = re.compile(r"^\s*<reasoning>.*?</reasoning>\s*<summary>.*?</summary>\s*$", re.DOTALL)
    if full_pattern.search(generated_text):
        return 1.0 # Perfect structure

    # Assign partial credit for having tags present
    has_reasoning = '<reasoning>' in generated_text and '</reasoning>' in generated_text
    has_summary = '<summary>' in generated_text and '</summary>' in generated_text

    if not has_reasoning and not has_summary:
        return 0.0

    if has_reasoning and has_summary:
        try:
            # Penalize for wrong order
            if generated_text.index('<reasoning>') > generated_text.index('<summary>'):
                return 0.2
        except ValueError:
            pass # Should not happen if both tags are present

    # Partial credit for having at least one complete tag pair
    return 0.5

def calculate_quality_reward(generated_text: str, reference_answer: str) -> float:
    """
    Calculates the ROUGE-L F1 score between the generated and reference summary.
    A higher reward is better (max 1.0).
    """
    summary_match = re.search(r'<summary>(.*?)</summary>', generated_text, re.DOTALL)
    if not summary_match:
        return 0.0  # No summary, no quality

    generated_summary = summary_match.group(1).strip()
    if not generated_summary:
        return 0.0

    scores = rouge_scorer_instance.score(reference_answer, generated_summary)
    return scores['rougeL'].fmeasure

# ==============================================================================
# Step 3: Create BATCH-COMPATIBLE Wrappers for the Trainer
# These functions will be passed to GRPOTrainer.
# ==============================================================================

def length_reward_batch(completions: List[str], **kwargs) -> List[float]:
    """Calculates length rewards for a batch of completions."""
    target_word_counts = kwargs["target_word_count"]
    target_sentence_counts = kwargs["target_sentence_count"]

    return [
        calculate_length_reward(
            generated_text=comp,
            target_word_count=t_wc,
            target_sentence_count=t_sc
        ) for comp, t_wc, t_sc in zip(completions, target_word_counts, target_sentence_counts)
    ]

def structure_reward_batch(completions: List[str], **kwargs) -> List[float]:
    """Calculates structure rewards for a batch of completions."""
    return [calculate_structure_reward(c) for c in completions]

def quality_reward_batch(completions: List[str], **kwargs) -> List[float]:
    """Calculates ROUGE-L quality rewards for a batch of completions."""
    reference_answers = kwargs["answer"]

    return [
        calculate_quality_reward(
            generated_text=comp,
            reference_answer=ref
        ) for comp, ref in zip(completions, reference_answers)
    ]

MAX_PROMPT_LENGTH = max_seq_length - max_completion_length
COMPLETION_CEILING = max_completion_length

training_args = GRPOConfig(
    #run_name=run_name,
    fp16=False,
    report_to="wandb",
    output_dir=output_dir,
    use_vllm=False,
    vllm_gpu_memory_utilization=0.3,
    learning_rate=learning_rate,
    max_steps=8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=gradient_accumulation_steps,
    logging_steps=logging_steps,
    save_steps=save_steps,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    max_grad_norm=0.1,
    num_generations=2,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=COMPLETION_CEILING,
)
trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[
        length_reward_batch,
        structure_reward_batch,
        quality_reward_batch
    ],
    #reward_weights=[
        #0.5,  # Weight for length
        #0.3,  # Weight for structure
        #0.2   # Weight for quality (ROUGE-L)
    #],
    args=training_args,
    train_dataset=dataset_dict['train'],      # or dataset_dict['train']
    eval_dataset=dataset_dict['validation'],         # or dataset_dict['validation']
)
print(f"Starting training...")
trainer.train()
print("Training finished.")

In [9]:
# --- Load Model and Tokenizer ---
print(f"Loading base model '{BASE_MODEL_NAME}'...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True, fast_inference=True, max_lora_rank=LORA_RANK,
    gpu_memory_utilization = 0.95
)


model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = LORA_RANK,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)


Loading base model 'HuggingFaceTB/SmolLM2-360M-Instruct'...
Unsloth: Patching vLLM to enable standby.
Unsloth: Patching vLLM v1 graph capture
Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.9.4: Fast Llama patching. Transformers: 4.55.4. vLLM: 0.10.1.1.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading HuggingFaceTB/SmolLM2-360M-Instruct with actual GPU utilization = 94.46%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 79.32 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 4096. Num Sequences = 368.
Unsloth: vLLM's KV Cache can use up to 74.22 GB. Also swap space = 6 GB.
Unsloth: 

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

HuggingFaceTB/SmolLM2-360M-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.


Unsloth 2025.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
#text = tokenizer.apply_chat_template([
#   {"role" : "system", "content" : SYSTEM_PROMPT},
#   {"role" : "user", "content" : "Calculate pi."},
#, tokenize = False, add_generation_prompt = True)

prompts = tokenizer.apply_chat_template(test_dataset[200]['prompt'], tokenize=False, add_generation_prompt=True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.1,
    #top_p = 0.95,
    max_tokens = 4096,
)
output = model.fast_generate(
    prompts,
    sampling_params = sampling_params,
    #lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

In [10]:
test_dataset

Dataset({
    features: ['target_word_count', 'target_sentence_count', 'prompt', 'answer', 'prompt_length'],
    num_rows: 571
})

In [13]:
# --- 1. Evaluate BASE model (without LoRA adapter) ---
print("\n" + "="*50 + "\nEvaluating BASE model...\n" + "="*50)
base_metrics, base_table = evaluate_model(
    model=model, tokenizer=tokenizer, test_dataset=test_dataset,
    lora_request=None, batch_size=BATCH_SIZE
)
wandb.log({"base/evaluation_table": base_table})
wandb.log({f"base/{k}": v for k, v in base_metrics.items()})
print("\nBase Model Metrics:")
for key, value in base_metrics.items(): print(f"  {key}: {value:.4f}")


Evaluating BASE model...


Evaluating Batches:   0%|          | 0/72 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

...

Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Base Model Metrics:
  rouge1: 0.1003
  rouge2: 0.0040
  rougeL: 0.0814
  avg_word_count_score: -0.4494
  avg_sentence_count_score: -0.5291
  avg_structure_score: -0.9419
  format_failure_rate: 0.0053


In [7]:
model_trained, tokenizer_trained = FastLanguageModel.from_pretrained(
    model_name=HF_REPO_ID, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True, fast_inference=True, max_lora_rank=LORA_RANK,
)

Unsloth: Patching vLLM v1 graph capture
Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.9.4: Fast Llama patching. Transformers: 4.55.4. vLLM: 0.10.1.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading HuggingFaceTB/SmolLM2-360M-Instruct with actual GPU utilization = 49.43%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 4096. Num Sequences = 288.
Unsloth: vLLM's KV Cache can use up to 18.86 GB. Also swap space = 6 GB.
Unsloth: Not an error, but `device` is not supported in vLLM. Skipping.
INFO 09-13 17:39:43 [utils.py:326] non-

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 09-13 17:40:00 [punica_selector.py:19] Using PunicaWrapperGPU.
INFO 09-13 17:40:01 [gpu_model_runner.py:2007] Model loading took 0.2616 GiB and 1.059807 seconds
INFO 09-13 17:40:18 [backends.py:548] Using cache directory: /root/.cache/vllm/torch_compile_cache/7d2b7111b5/rank_0_0/backbone for vLLM's torch.compile
INFO 09-13 17:40:18 [backends.py:559] Dynamo bytecode transform time: 15.99 s
INFO 09-13 17:40:28 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 8.736 s
INFO 09-13 17:40:31 [monitor.py:34] torch.compile takes 15.99 s in total
INFO 09-13 17:40:33 [gpu_worker.py:276] Available KV cache memory: 18.75 GiB
INFO 09-13 17:40:34 [kv_cache_utils.py:849] GPU KV cache size: 491,568 tokens
INFO 09-13 17:40:34 [kv_cache_utils.py:853] Maximum concurrency for 4,096 tokens per request: 120.01x
INFO 09-13 17:40:34 [vllm_utils.py:667] Unsloth: Running patched vLLM v1 `capture_model`.


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:27<00:00,  2.45it/s]

INFO 09-13 17:41:01 [gpu_model_runner.py:2708] Graph capturing finished in 27 secs, took 1.28 GiB
INFO 09-13 17:41:01 [vllm_utils.py:674] Unsloth: Patched vLLM v1 graph capture finished in 27 secs.


INFO 09-13 17:41:03 [core.py:214] init engine (profile, create kv cache, warmup model) took 61.70 seconds
INFO 09-13 17:41:03 [llm.py:298] Supported_tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'k_norm', 'post_feedforward_layernorm', 'q_norm']
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'k_norm', 'post_feedforward_layernorm', 'q_norm']
HuggingFaceTB/SmolLM2-360M-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.


Unsloth 2025.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [14]:
model_trained.save_lora("grpo_saved_lora")

In [9]:
# @title Step 6: Load Model and Run Evaluation
# --- 2. Evaluate TRAINED model (with LoRA adapter) ---
print("\n" + "="*50 + f"\nEvaluating TRAINED model with adapter from '{HF_REPO_ID}'...\n" + "="*50)
#lora_request = model.load_lora(HF_REPO_ID)
trained_metrics, trained_table = evaluate_model(
    model=model_trained, tokenizer=tokenizer_trained, test_dataset=test_dataset,
    lora_request=model_trained.load_lora("grpo_saved_lora"), batch_size=BATCH_SIZE
)
wandb.log({"trained/evaluation_table": trained_table})
wandb.log({f"trained/{k}": v for k, v in trained_metrics.items()})
print("\nTrained Model Metrics:")
for key, value in trained_metrics.items(): print(f"  {key}: {value:.4f}")

# --- Finish W&B Run ---
wandb.finish()
print("\nEvaluation complete. Results logged to W&B.")


Evaluating TRAINED model with adapter from 'M4TT1A/my-llama3-grpo-lora'...


Evaluating Batches:   0%|          | 0/72 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

...

Adding requests:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Trained Model Metrics:
  rouge1: 0.1047
  rouge2: 0.0049
  rougeL: 0.0825
  avg_word_count_score: -0.4529
  avg_sentence_count_score: -0.5174
  avg_structure_score: -0.9381
  format_failure_rate: 0.0053


trained/avg_sentence_count_score,▁
trained/avg_structure_score,▁
trained/avg_word_count_score,▁
trained/format_failure_rate,▁
trained/rouge1,▁
trained/rouge2,▁
trained/rougeL,▁
trained/avg_sentence_count_score,-0.5174
trained/avg_structure_score,-0.93809
trained/avg_word_count_score,-0.45289
trained/format_failure_rate,0.00525



Evaluation complete. Results logged to W&B.


In [9]:
# @title Step 6: Load Model and Run Evaluation
# --- 2. Evaluate TRAINED model (with LoRA adapter) ---
print("\n" + "="*50 + f"\nEvaluating TRAINED model with adapter from '{HF_REPO_ID}'...\n" + "="*50)
#lora_request = model.load_lora(HF_REPO_ID)
trained_metrics, trained_table = evaluate_model(
    model=model, tokenizer=tokenizer, test_dataset=test_dataset,
    lora_request=model.load_lora("grpo_saved_lora"), batch_size=BATCH_SIZE
)
wandb.log({"trained/evaluation_table": trained_table})
wandb.log({f"trained/{k}": v for k, v in trained_metrics.items()})
print("\nTrained Model Metrics:")
for key, value in trained_metrics.items(): print(f"  {key}: {value:.4f}")

# --- Finish W&B Run ---
wandb.finish()
print("\nEvaluation complete. Results logged to W&B.")


Evaluating TRAINED model with adapter from 'M4TT1A/my-llama3-grpo-lora'...


Evaluating Batches:   0%|          | 0/72 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

...

Adding requests:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Trained Model Metrics:
  rouge1: 0.0964
  rouge2: 0.0042
  rougeL: 0.0758
  avg_word_count_score: -0.4572
  avg_sentence_count_score: -0.5246
  avg_structure_score: -0.9522
  format_failure_rate: 0.0053


trained/avg_sentence_count_score,▁
trained/avg_structure_score,▁
trained/avg_word_count_score,▁
trained/format_failure_rate,▁
trained/rouge1,▁
trained/rouge2,▁
trained/rougeL,▁
trained/avg_sentence_count_score,-0.52464
trained/avg_structure_score,-0.95219
trained/avg_word_count_score,-0.4572
trained/format_failure_rate,0.00525



Evaluation complete. Results logged to W&B.


In [ ]:
formatted_prompt = tokenizer.apply_chat_template(
    test_dataset[0]['prompt'],
    tokenize = False,
    add_generation_prompt = True,
)

# --- Generate the completion ---
inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
#model.load_adapter(HF_REPO_ID)

outputs = model.generate(
    **inputs,
    max_new_tokens = 1000,
    do_sample = False, # Use greedy decoding for a deterministic check
    pad_token_id = tokenizer.eos_token_id
)

# --- Decode and print the result ---
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n" + "="*50)
print("PROMPT SENT TO MODEL:")
print("="*50)
print(formatted_prompt)
print("\n" + "="*50)
print("MODEL GENERATION:")
print("="*50)
# We slice the result to only show the newly generated part
print(result[len(formatted_prompt):].strip())
print("="*50)

In [14]:
print(result[len(formatted_prompt):].strip())


is authorized under the Metropolitan Medical Response System Program Act of 2009 (6 U.S.C. 311). The Act provides funding for the program, including financial assistance, consultation, and performance metrics.

The Metropolitan Medical Response System Program (MMRS) is authorized to review the program under the provisions of the Homeland Security Act of 2002 (6 U.S.C. 311). The Act requires the Administrator and the Assistant Secretary, Office of Health Affairs, to submit a report on the results of the review under this subsection.

The Act also requires the use of funds under the program to support the integration of emergency management, health, and medical systems into a coordinated response to mass casualty incidents caused by natural disasters, acts of terrorism, and other man-made disasters.

The Act provides for financial assistance to support the development and maintenance of an initial pharmaceutical stockpile sufficient to protect first responders, their families, and immedi

In [ ]:
    sampling_params = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=1024)
    rouge = evaluate.load('rouge')

    results_data = []
    all_generated_summaries, all_gold_summaries = [], []
    all_word_scores, all_sentence_scores, all_structure_scores = [], [], []

    for i in tqdm(range(0, len(test_dataset), batch_size), desc="Evaluating Batches"):
        batch = test_dataset[i:i+batch_size]

        # Access the entire column (list of prompts) at once
        batch_prompts_data = batch['prompt']

        # Apply the chat template to each item in the list of prompts
        prompts = [
            tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)
            for chat_history in batch_prompts_data
        ]


outputs = model.fast_generate(prompts, sampling_params=sampling_params, lora_request=lora_request)

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

Unsloth: Patching Xformers to fix some performance issues.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 03-07 09:37:36 __init__.py:207] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.8: Fast Llama patching. Transformers: 4.48.3. vLLM: 0.7.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit with actual GPU utilization = 59.43%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1024. Num Se

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

INFO 03-07 09:40:35 model_runner.py:1562] Graph capturing finished in 43 secs, took 0.59 GiB
INFO 03-07 09:40:35 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 60.26 seconds


tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth 2025.3.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:51<00:00, 51.81s/it, est. speed input: 0.75 toks/s, output: 17.01 toks/s]


'**Calculating Pi using Python**\n\nPi (π) is a mathematical constant representing the ratio of a circle\'s circumference to its diameter. Here\'s a simple and efficient way to calculate an approximation of pi using Python.\n\n### Using the Monte Carlo Method\n\nThe Monte Carlo method is a computational algorithm that uses random sampling to approximate a value. In this case, we can use it to estimate pi by generating random points within a square and checking if they fall inside a quarter-circle inscribed within it.\n\n```python\nimport random\nimport math\n\ndef estimate_pi(num_samples):\n    """\n    Estimate the value of pi using the Monte Carlo method.\n\n    Args:\n    num_samples (int): The number of random points to generate.\n\n    Returns:\n    float: An approximation of pi.\n    """\n    points_inside_circle = 0\n\n    for _ in range(num_samples):\n        x, y = random.random(), random.random()\n        distance = x**2 + y**2\n        if distance <= 1:\n            points_i

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:23<00:00, 23.17s/it, est. speed input: 2.63 toks/s, output: 15.80 toks/s]


'Calculating pi to a high degree of accuracy is a complex task that requires a large amount of computational power. However, I can provide you with an approximate value of pi or show you a simple method to calculate it.\n\nOne of the simplest methods to calculate pi is the Leibniz formula, which is an infinite series:\n\npi/4 = 1 - 1/3 + 1/5 - 1/7 + 1/9 - ...\n\nThis series can be used to calculate an approximation of pi.\n\n<reasoning>\npi = 4 * (1 - 1/3 + 1/5 - 1/7 + 1/9 - ...)\n</reasoning>\n\nThis is a simple, yet effective method to calculate pi. However, the more terms you use, the more accurate the result will be.\n\nTo calculate pi to a high degree of accuracy, you would need to use a computer program to perform the calculation.\n\n<answer>\n3.141592653589793 (approximately)\n</answer>\n\nFor a more accurate result, I can provide you with a Python code snippet to calculate pi:\n\n```python\nimport math\n\ndef calculate_pi(n):\n    pi = 0.0\n    for i in range(n):\n        pi +=

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
